<a href="https://colab.research.google.com/github/NikitVaishnav/GenAI/blob/main/GenAI_P4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

corpus_path = Path("Int.txt")
if not corpus_path.is_file():
    # Generate a sample input.txt automatically so the notebook runs immediately
    sample_words = [
        "hello", "world", "sequence", "learning", "tensorflow",
        "keras", "encoder", "decoder", "recurrent", "network"
    ]
    corpus_path.write_text("\n".join(sample_words), encoding="utf-8")
    print("Created a sample input.txt with default words.")

lines = [line.strip().lower() for line in corpus_path.read_text(encoding="utf-8").splitlines()]
character_sequences = [line for line in lines if len(line) >= 3]

# If only one line is provided, duplicate it so that validation split doesn't crash
if len(character_sequences) == 1:
    character_sequences = character_sequences * 2
elif len(character_sequences) < 1:
    raise ValueError("The corpus must contain at least one non-empty line of length >= 3.")

characters = sorted(set("".join(character_sequences)))
char_to_id = {character: index for index, character in enumerate(characters, start=1)}
id_to_char = {index: character for character, index in char_to_id.items()}

PAD_ID = 0
START_ID = len(char_to_id) + 1
END_ID = START_ID + 1
VOCAB_SIZE = END_ID + 1

max_input_length = max(map(len, character_sequences))
encoder_data = pad_sequences(
    [[char_to_id[character] for character in sequence] for sequence in character_sequences],
    maxlen=max_input_length,
    padding="post",
    dtype="int32",
)

target_sequences = [
    [START_ID]
    + [char_to_id[character] for character in sequence[::-1]]
    + [END_ID]
    for sequence in character_sequences
]
decoder_length = max_input_length + 1
decoder_input_data = pad_sequences(
    [sequence[:-1] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
decoder_target_data = pad_sequences(
    [sequence[1:] for sequence in target_sequences],
    maxlen=decoder_length,
    padding="post",
    dtype="int32",
)
target_weights = (decoder_target_data != PAD_ID).astype("float32")

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(encoder_data))
validation_size = max(1, int(0.2 * len(indices))) if len(indices) > 1 else 0
validation_indices = indices[:validation_size] if validation_size > 0 else indices
training_indices = indices[validation_size:] if validation_size > 0 else indices

print(f"Examples: {len(character_sequences)}")
print(f"Character vocabulary (including reserved tokens): {VOCAB_SIZE}")
print(f"Maximum input length: {max_input_length}")